<div style="padding: 28px 32px; border-radius: 18px; background: linear-gradient(135deg, #1f2937 0%, #b91c1c 55%, #fb7185 100%); color: white; box-shadow: 0 20px 50px rgba(31, 41, 55, 0.18);">
  <p style="margin: 0; font-size: 14px; letter-spacing: 0.18em; text-transform: uppercase; opacity: 0.85;">Customer Intelligence • Revenue Forecasting System</p>
  <h1 style="margin: 12px 0 10px; font-size: 34px;">Churn-Risk Analysis</h1>
  <p style="margin: 0; font-size: 17px; max-width: 840px; line-height: 1.7;">
    This notebook builds a proxy churn-risk model from transaction history. Since the dataset does not contain an explicit churn flag,
    churn-risk is defined behaviorally using customer recency.
  </p>
</div>


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import ConfusionMatrixDisplay

pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", context="talk")

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.data_utils import load_cleaned_data
from scripts.churn_utils import build_customer_features, run_churn_modeling

df = load_cleaned_data(project_root)
customer_features = build_customer_features(df, churn_threshold_days=90)
customer_features.head()


## Churn Logic

In this project, a customer is labeled as **churn-risk** if their recency exceeds 90 days. This is a proxy churn definition based on inactivity rather than a contractual cancellation event.


In [ ]:
customer_features["churn_risk"].value_counts().rename(index={0: "Active", 1: "Churn Risk"})


## Train and Evaluate Baseline Models


In [ ]:
model_results = run_churn_modeling(customer_features, project_root)

metrics_df = pd.DataFrame(model_results["metrics"]).T.sort_values("recall", ascending=False)
metrics_df


In [ ]:
best_model_name = model_results["best_model_name"]
predictions = model_results["predictions"]
confusion_matrix = model_results["confusion_matrix"]
feature_importance = model_results["feature_importance"]

print(f"Best model selected: {best_model_name}")
predictions.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sns.histplot(predictions["churn_probability"], bins=25, kde=True, color="#b91c1c", ax=axes[0])
axes[0].set_title("Distribution of Churn Probability")
axes[0].set_xlabel("Predicted Churn Probability")

ConfusionMatrixDisplay(confusion_matrix=confusion_matrix).plot(ax=axes[1], colorbar=False)
axes[1].set_title(f"Confusion Matrix - {best_model_name}")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance.head(10), x="importance", y="feature", hue="feature", legend=False, palette="Reds_r")
plt.title("Top Churn Drivers")
plt.xlabel("Relative Importance")
plt.ylabel("")
plt.tight_layout()
plt.show()


> **Insight:** This churn output is a behavioral risk signal, not a contractual churn label. That makes it especially useful for retention monitoring, prioritizing outreach, and surfacing which customers should be reviewed first.


## Save Churn Predictions


In [ ]:
churn_output = project_root / "outputs" / "csv" / "churn_predictions.csv"
churn_output.parent.mkdir(parents=True, exist_ok=True)
predictions.to_csv(churn_output, index=False)
print(f"Saved churn predictions to: {churn_output.resolve()}")
